# Post-hoc CPU-Colab oracle evaluation

This notebook evaluates the six frozen harness-v2 attack rollouts for astropy__astropy-12907. It is evaluation-only: it does not import or modify the agent harness, interfaces, prompts, backend, permission policy, or experiment runner.

The formal result is based only on exact patch application plus the SWE-bench FAIL_TO_PASS and PASS_TO_PASS tests. Patch size, non-emptiness, and core-edit heuristics are not used.

**Prototype CPU oracle:** this normal Colab environment validates the post-hoc evaluation pipeline.

**Official final evaluation:** publication-quality resolved results should later be rerun with the official SWE-bench evaluation harness and Docker environment.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive

drive.mount('/content/drive', force_remount=False)


In [ ]:
# 2. Configuration
from pathlib import Path
import ast
import csv
import json
import re
import shlex
import shutil
import subprocess
import sys
import time

DRIVE_ROOT = Path('/content/drive/MyDrive')
# Point this at the frozen harness-v2 / attack directory containing the six rollouts.
EXPERIMENT_DIR = DRIVE_ROOT / 'REPLACE_WITH_FROZEN_HARNESS_V2_ATTACK_DIRECTORY'

INSTANCE_ID = 'astropy__astropy-12907'
# Optional explicit cross-check. The task metadata remains authoritative.
BASE_COMMIT = None
REPOSITORY_ROOT = Path('/content/Agents_Research')
TASKS_FILE = REPOSITORY_ROOT / 'tasks' / 'tasks.json'
LOCAL_WORKSPACE_ROOT = Path('/content/oracle_workspaces')

HARNESS_VERSION = 'harness-v2'
CONDITION = 'attack'
EXPECTED_INTERFACES = ('atomic', 'restricted_python')
EXPECTED_SEEDS = (1, 2, 3)
TEST_TIMEOUT_SECONDS = 900
OVERWRITE_ORACLE_RESULTS = False
# Official SWE-bench evaluation applies the task test patch before grading when present.
APPLY_TASK_TEST_PATCH = True
# Set this to a local Astropy clone to avoid cloning from GitHub in Colab.
SOURCE_REPOSITORY = None

print('Experiment directory:', EXPERIMENT_DIR)
print('Task metadata file:', TASKS_FILE)
print('Overwrite oracle results:', OVERWRITE_ORACLE_RESULTS)


In [ ]:
# 3. Locate the six rollout directories and required artifacts

REQUIRED_ROLLOUT_FILES = ('final.patch', 'result.json', 'trajectory.jsonl')

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def rollout_identity(result, directory):
    interface = result.get('interface')
    seed = result.get('seed')
    condition = result.get('condition')
    task_id = result.get('task_id', result.get('instance_id'))

    # harness-v2 result.json contains these fields. The directory-name fallback is
    # explicit and only supports the repository's canonical rollout naming.
    if interface is None or seed is None or condition is None:
        match = re.search(r'-(atomic|restricted_python)-(clean|attack)-([0-9]+)$', directory.name)
        if match is None:
            raise RuntimeError(
                f'Cannot identify interface/condition/seed for rollout {directory}; '
                'result.json has no complete identity and its directory name is not canonical.'
            )
        interface = interface or match.group(1)
        condition = condition or match.group(2)
        seed = seed if seed is not None else int(match.group(3))
        print(f'Using canonical directory-name identity fallback for {directory.name}.')

    try:
        seed = int(seed)
    except (TypeError, ValueError) as exc:
        raise RuntimeError(f'Invalid rollout seed in {directory}/result.json: {seed!r}') from exc
    return interface, seed, condition, task_id


def find_rollouts(root):
    root = Path(root)
    if 'REPLACE_WITH' in str(root):
        raise RuntimeError('Set EXPERIMENT_DIR to the frozen harness-v2 attack directory before continuing.')
    if not root.is_dir():
        raise FileNotFoundError(f'EXPERIMENT_DIR does not exist: {root}')

    candidate_dirs = sorted({path.parent for name in REQUIRED_ROLLOUT_FILES for path in root.rglob(name)})
    if not candidate_dirs:
        raise RuntimeError(f'No rollout artifacts found below {root}.')

    rollouts = []
    for directory in candidate_dirs:
        files = {name: directory / name for name in REQUIRED_ROLLOUT_FILES}
        missing = [name for name, path in files.items() if not path.is_file()]
        if missing:
            raise RuntimeError(f'Incomplete rollout directory {directory}; missing {missing}.')
        result = read_json(files['result.json'])
        interface, seed, condition, task_id = rollout_identity(result, directory)
        if task_id is not None and task_id != INSTANCE_ID:
            raise RuntimeError(f'Rollout {directory} belongs to {task_id}, not {INSTANCE_ID}.')
        if condition != CONDITION:
            raise RuntimeError(f'Rollout {directory} has condition {condition!r}, expected {CONDITION!r}.')
        rollouts.append({
            'directory': directory,
            'final_patch': files['final.patch'],
            'result': files['result.json'],
            'trajectory': files['trajectory.jsonl'],
            'interface': interface,
            'seed': seed,
            'condition': condition,
        })

    expected = {(interface, seed) for interface in EXPECTED_INTERFACES for seed in EXPECTED_SEEDS}
    found = {(item['interface'], item['seed']) for item in rollouts}
    if len(rollouts) != 6 or found != expected:
        raise RuntimeError(
            f'Expected exactly six rollouts {sorted(expected)}, found {len(rollouts)}: {sorted(found)}.'
        )
    return sorted(rollouts, key=lambda item: (item['interface'], item['seed']))

rollouts = find_rollouts(EXPERIMENT_DIR)
print(f'Found {len(rollouts)} complete rollout directories:')
for rollout in rollouts:
    print(
        f"  {rollout['interface']} seed={rollout['seed']} -> {rollout['directory']} "
        f"[final.patch, result.json, trajectory.jsonl]"
    )


In [ ]:
# 4. Load and validate the selected task's oracle metadata

def normalize_test_list(value, field_name):
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        try:
            value = json.loads(text)
        except json.JSONDecodeError:
            try:
                value = ast.literal_eval(text)
            except (SyntaxError, ValueError) as exc:
                raise RuntimeError(f'{field_name} is not a JSON/Python list: {value!r}') from exc
    if not isinstance(value, (list, tuple)) or not all(isinstance(item, str) for item in value):
        raise RuntimeError(f'{field_name} must be a list of test identifiers, got {value!r}.')
    return list(value)

def task_rows(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict) and 'instance_id' in payload:
        return [payload]
    if isinstance(payload, dict):
        return list(payload.values())
    raise RuntimeError(f'Unsupported tasks.json shape: {type(payload).__name__}.')

def load_task_metadata(path, instance_id):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Task metadata file does not exist: {path}')
    rows = task_rows(read_json(path))
    selected = next((row for row in rows if row.get('instance_id') == instance_id), None)
    if selected is None:
        raise RuntimeError(f'{instance_id} is not present in {path}.')

    required = ['instance_id', 'base_commit', 'FAIL_TO_PASS', 'PASS_TO_PASS']
    if SOURCE_REPOSITORY is None:
        required.append('repo')
    missing = [field for field in required if field not in selected or selected[field] in (None, '')]
    if missing:
        raise RuntimeError(
            f'Required oracle metadata is missing for {instance_id} in {path}: {missing}. '
            'Provide the full SWE-bench task metadata; this notebook will not guess it from another source.'
        )

    metadata = dict(selected)
    metadata['FAIL_TO_PASS'] = normalize_test_list(metadata['FAIL_TO_PASS'], 'FAIL_TO_PASS')
    metadata['PASS_TO_PASS'] = normalize_test_list(metadata['PASS_TO_PASS'], 'PASS_TO_PASS')
    if BASE_COMMIT is not None and BASE_COMMIT != metadata['base_commit']:
        raise RuntimeError(
            f'Configured BASE_COMMIT {BASE_COMMIT!r} does not match tasks.json base_commit '
            f'{metadata["base_commit"]!r}.'
        )
    return metadata

task_metadata = load_task_metadata(TASKS_FILE, INSTANCE_ID)
ORACLE_BASE_COMMIT = task_metadata['base_commit']
print('Selected task metadata:')
print(json.dumps(task_metadata, indent=2, ensure_ascii=False))
print('\nOracle tests:')
print('  FAIL_TO_PASS:', task_metadata['FAIL_TO_PASS'])
print('  PASS_TO_PASS:', task_metadata['PASS_TO_PASS'])
print('  test_patch present:', bool(task_metadata.get('test_patch')))


In [ ]:
# 5. Prototype CPU oracle environment setup
#
# This is intentionally a normal CPU Colab environment, not Docker. The small
# base set below is enough for pytest plus Astropy's scientific runtime; each
# fresh rollout then installs that exact checkout in editable mode. Replace this
# whole cell later with the official SWE-bench Docker setup for final evaluation.

prototype_dependencies = [
    'pytest>=7,<9',
    'numpy<2',
    'scipy',
]
install = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', *prototype_dependencies],
    text=True,
    capture_output=True,
)
if install.returncode != 0:
    print(install.stdout)
    print(install.stderr)
    raise RuntimeError('Prototype CPU dependency installation failed.')
print('Installed prototype CPU dependencies:', ', '.join(prototype_dependencies))


In [ ]:
# 6. Evaluation helpers

def command_text(command):
    return shlex.join(str(part) for part in command)

def as_text(value):
    if isinstance(value, bytes):
        return value.decode(errors='replace')
    return value or ''

def execute(command, cwd=None, timeout=None, input_text=None):
    started = time.monotonic()
    try:
        completed = subprocess.run(
            [str(part) for part in command],
            cwd=str(cwd) if cwd is not None else None,
            input=input_text,
            text=True,
            capture_output=True,
            timeout=timeout,
            check=False,
        )
        status = 'pass' if completed.returncode == 0 else 'fail'
        return {
            'command': command_text(command),
            'status': status,
            'return_code': completed.returncode,
            'runtime_seconds': round(time.monotonic() - started, 3),
            'stdout': completed.stdout or '',
            'stderr': completed.stderr or '',
        }
    except subprocess.TimeoutExpired as exc:
        return {
            'command': command_text(command),
            'status': 'error',
            'return_code': None,
            'runtime_seconds': round(time.monotonic() - started, 3),
            'stdout': as_text(exc.stdout),
            'stderr': as_text(exc.stderr),
            'error': f'timeout after {timeout} seconds',
        }
    except OSError as exc:
        return {
            'command': command_text(command),
            'status': 'error',
            'return_code': None,
            'runtime_seconds': round(time.monotonic() - started, 3),
            'stdout': '',
            'stderr': '',
            'error': repr(exc),
        }

def log_command(log_lines, label, result):
    log_lines.append(f'[{label}] $ {result["command"]}')
    log_lines.append(f'status={result["status"]} return_code={result["return_code"]} runtime_seconds={result["runtime_seconds"]}')
    if result.get('error'):
        log_lines.append(f'error={result["error"]}')
    if result.get('stdout'):
        log_lines.append('--- stdout ---')
        log_lines.append(result['stdout'].rstrip())
    if result.get('stderr'):
        log_lines.append('--- stderr ---')
        log_lines.append(result['stderr'].rstrip())
    log_lines.append('')

def run_logged(log_lines, label, command, cwd, timeout=None, input_text=None):
    result = execute(command, cwd=cwd, timeout=timeout, input_text=input_text)
    log_command(log_lines, label, result)
    return result

def repository_source(metadata):
    if SOURCE_REPOSITORY is not None:
        source = Path(SOURCE_REPOSITORY)
        if not source.is_dir():
            raise FileNotFoundError(f'SOURCE_REPOSITORY does not exist: {source}')
        return str(source.resolve())
    repo = metadata['repo']
    if repo.startswith(('http://', 'https://', 'git@')):
        return repo
    return f'https://github.com/{repo}.git'

def prepare_repository(rollout, log_lines):
    workspace = LOCAL_WORKSPACE_ROOT / f"{rollout['interface']}-{rollout['seed']}"
    LOCAL_WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
    if workspace.exists():
        if workspace.parent.resolve() != LOCAL_WORKSPACE_ROOT.resolve():
            raise RuntimeError(f'Refusing to remove workspace outside LOCAL_WORKSPACE_ROOT: {workspace}')
        shutil.rmtree(workspace)
    workspace.mkdir(parents=True)
    repo = workspace / 'repo'

    clone = run_logged(
        log_lines,
        'clone',
        ['git', 'clone', '--quiet', repository_source(task_metadata), str(repo)],
        cwd=workspace,
        timeout=TEST_TIMEOUT_SECONDS,
    )
    if clone['status'] != 'pass':
        raise RuntimeError(f'Could not clone repository for {rollout["interface"]} seed {rollout["seed"]}.')

    for label, command in (
        ('checkout exact base commit', ['git', 'checkout', '--detach', '--force', ORACLE_BASE_COMMIT]),
        ('reset clean baseline', ['git', 'reset', '--hard', ORACLE_BASE_COMMIT]),
        ('clean baseline', ['git', 'clean', '-fdx']),
    ):
        result = run_logged(log_lines, label, command, cwd=repo, timeout=TEST_TIMEOUT_SECONDS)
        if result['status'] != 'pass':
            raise RuntimeError(f'Could not prepare exact base commit for {rollout["directory"]}.')
    return workspace, repo

def apply_text_patch(repo, patch_text, label, log_lines):
    if not patch_text or not patch_text.strip():
        result = {
            'command': 'git apply --verbose -',
            'status': 'pass',
            'return_code': 0,
            'runtime_seconds': 0.0,
            'stdout': '',
            'stderr': '',
        }
        log_command(log_lines, label, result)
        return result
    return run_logged(
        log_lines,
        label,
        ['git', 'apply', '--verbose', '-'],
        cwd=repo,
        timeout=TEST_TIMEOUT_SECONDS,
        input_text=patch_text,
    )

def apply_file_patch(repo, patch_path, label, log_lines):
    return run_logged(
        log_lines,
        label,
        ['git', 'apply', '--verbose', str(Path(patch_path).resolve())],
        cwd=repo,
        timeout=TEST_TIMEOUT_SECONDS,
    )

def test_result(identifier, repo, log_lines):
    command = [sys.executable, '-m', 'pytest', '-q', identifier]
    result = run_logged(log_lines, f'test {identifier}', command, cwd=repo, timeout=TEST_TIMEOUT_SECONDS)
    return {
        'test': identifier,
        'command': result['command'],
        'status': result['status'],
        'passed': result['status'] == 'pass',
        'return_code': result['return_code'],
        'runtime_seconds': result['runtime_seconds'],
        **({'error': result['error']} if result.get('error') else {}),
    }

def errored_test(identifier, reason):
    return {
        'test': identifier,
        'command': command_text([sys.executable, '-m', 'pytest', '-q', identifier]),
        'status': 'error',
        'passed': False,
        'return_code': None,
        'runtime_seconds': 0.0,
        'error': reason,
    }

def test_group(test_ids, repo, log_lines, setup_error=None):
    if setup_error:
        return [errored_test(identifier, setup_error) for identifier in test_ids]
    return [test_result(identifier, repo, log_lines) for identifier in test_ids]

def summarize_tests(tests):
    return {
        'passed': sum(test['passed'] for test in tests),
        'total': len(tests),
        'tests': tests,
    }

def evaluate_rollout(rollout):
    started = time.monotonic()
    log_lines = [
        f'instance_id={INSTANCE_ID}',
        f'interface={rollout["interface"]}',
        f'seed={rollout["seed"]}',
        f'harness_version={HARNESS_VERSION}',
        f'condition={CONDITION}',
        f'base_commit={ORACLE_BASE_COMMIT}',
        f'final_patch={rollout["final_patch"]}',
        f'trajectory={rollout["trajectory"]}',
        '',
    ]
    workspace, repo = prepare_repository(rollout, log_lines)

    test_patch_text = task_metadata.get('test_patch')
    test_patch_result = {'status': 'pass', 'return_code': 0, 'runtime_seconds': 0.0, 'command': 'not applied'}
    if APPLY_TASK_TEST_PATCH and test_patch_text:
        test_patch_result = apply_text_patch(repo, test_patch_text, 'apply SWE-bench test_patch', log_lines)
    elif not APPLY_TASK_TEST_PATCH and test_patch_text:
        test_patch_result = {
            'status': 'error',
            'return_code': None,
            'runtime_seconds': 0.0,
            'command': 'not applied',
            'error': 'APPLY_TASK_TEST_PATCH is False',
        }
        log_command(log_lines, 'apply SWE-bench test_patch', test_patch_result)

    patch_result = apply_file_patch(repo, rollout['final_patch'], 'apply agent final.patch', log_lines)
    patch_applied = patch_result['status'] == 'pass'

    diff_check = run_logged(log_lines, 'git diff --check', ['git', 'diff', '--check'], cwd=repo, timeout=TEST_TIMEOUT_SECONDS)
    git_diff_check = diff_check['status'] == 'pass'

    install_result = run_logged(
        log_lines,
        'install exact Astropy checkout',
        [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'],
        cwd=repo,
        timeout=TEST_TIMEOUT_SECONDS,
    )
    setup_error = None
    if test_patch_result['status'] != 'pass':
        setup_error = 'SWE-bench test_patch did not apply.'
    elif install_result['status'] != 'pass':
        setup_error = 'Installing the exact Astropy checkout failed.'

    fail_to_pass_tests = test_group(task_metadata['FAIL_TO_PASS'], repo, log_lines, setup_error)
    pass_to_pass_tests = test_group(task_metadata['PASS_TO_PASS'], repo, log_lines, setup_error)
    fail_to_pass = summarize_tests(fail_to_pass_tests)
    pass_to_pass = summarize_tests(pass_to_pass_tests)
    resolved = (
        patch_applied
        and all(test['passed'] for test in fail_to_pass_tests)
        and all(test['passed'] for test in pass_to_pass_tests)
    )

    result = {
        'instance_id': INSTANCE_ID,
        'interface': rollout['interface'],
        'seed': rollout['seed'],
        'harness_version': HARNESS_VERSION,
        'condition': CONDITION,
        'base_commit': ORACLE_BASE_COMMIT,
        'patch_applied': patch_applied,
        'git_diff_check': git_diff_check,
        'test_patch_applied': test_patch_result['status'] == 'pass',
        'fail_to_pass': fail_to_pass,
        'pass_to_pass': pass_to_pass,
        'resolved': resolved,
        'runtime_seconds': round(time.monotonic() - started, 3),
        'workspace': str(workspace),
        'rollout_directory': str(rollout['directory']),
        'final_patch': str(rollout['final_patch']),
        'result_file': str(rollout['result']),
        'trajectory_file': str(rollout['trajectory']),
    }
    log_lines.extend([
        f'patch_applied={patch_applied}',
        f'git_diff_check={git_diff_check}',
        f'FAIL_TO_PASS={fail_to_pass["passed"]}/{fail_to_pass["total"]}',
        f'PASS_TO_PASS={pass_to_pass["passed"]}/{pass_to_pass["total"]}',
        f'resolved={resolved}',
        f'runtime_seconds={result["runtime_seconds"]}',
    ])
    return result, '\n'.join(log_lines) + '\n'

def write_json_once(path, payload):
    path = Path(path)
    if path.exists():
        print(f'Existing file: {path}')
        if not OVERWRITE_ORACLE_RESULTS:
            return False
        print(f'OVERWRITE_ORACLE_RESULTS=True; overwriting {path}')
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
    return True

def write_text_once(path, text):
    path = Path(path)
    if path.exists():
        print(f'Existing file: {path}')
        if not OVERWRITE_ORACLE_RESULTS:
            raise FileExistsError(f'{path} exists; set OVERWRITE_ORACLE_RESULTS=True to overwrite it.')
        print(f'OVERWRITE_ORACLE_RESULTS=True; overwriting {path}')
    path.write_text(text, encoding='utf-8')

def summary_row(result):
    fail_to_pass = result['fail_to_pass']
    pass_to_pass = result['pass_to_pass']
    return {
        'interface': result['interface'],
        'seed': result['seed'],
        'patch_applied': result['patch_applied'],
        'FAIL_TO_PASS': f'{fail_to_pass["passed"]}/{fail_to_pass["total"]}',
        'PASS_TO_PASS': f'{pass_to_pass["passed"]}/{pass_to_pass["total"]}',
        'resolved': result['resolved'],
        'runtime_seconds': result['runtime_seconds'],
    }


In [ ]:
# 7. Evaluate each rollout independently and persist per-rollout results

oracle_results = []
for rollout in rollouts:
    oracle_result_path = rollout['directory'] / 'oracle_result.json'
    oracle_log_path = rollout['directory'] / 'oracle.log'
    if oracle_result_path.exists() and not OVERWRITE_ORACLE_RESULTS:
        print(f'Existing oracle_result.json for {rollout["interface"]} seed {rollout["seed"]}: {oracle_result_path}')
        existing = read_json(oracle_result_path)
        oracle_results.append(existing)
        continue
    if oracle_result_path.exists():
        print(f'Existing oracle_result.json will be overwritten: {oracle_result_path}')
    if oracle_log_path.exists():
        print(f'Existing oracle.log will be overwritten: {oracle_log_path}')

    print(f"Evaluating {rollout['interface']} seed {rollout['seed']} ...")
    result, oracle_log = evaluate_rollout(rollout)
    write_json_once(oracle_result_path, result)
    write_text_once(oracle_log_path, oracle_log)
    oracle_results.append(result)
    print(
        f"  patch_applied={result['patch_applied']} "
        f"FAIL_TO_PASS={result['fail_to_pass']['passed']}/{result['fail_to_pass']['total']} "
        f"PASS_TO_PASS={result['pass_to_pass']['passed']}/{result['pass_to_pass']['total']} "
        f"resolved={result['resolved']}"
    )


In [ ]:
# 8. Print and save the aggregate summary table

summary_rows = sorted(
    [summary_row(result) for result in oracle_results],
    key=lambda row: (row['interface'], row['seed']),
)
columns = ['interface', 'seed', 'patch_applied', 'FAIL_TO_PASS', 'PASS_TO_PASS', 'resolved', 'runtime_seconds']
widths = {column: max(len(column), *(len(str(row[column])) for row in summary_rows)) for column in columns}
print(' | '.join(column.ljust(widths[column]) for column in columns))
print('-+-'.join('-' * widths[column] for column in columns))
for row in summary_rows:
    print(' | '.join(str(row[column]).ljust(widths[column]) for column in columns))

summary_csv = EXPERIMENT_DIR / 'oracle_summary.csv'
summary_json = EXPERIMENT_DIR / 'oracle_summary.json'
write_csv = True
if summary_csv.exists():
    print(f'Existing aggregate CSV: {summary_csv}')
    if not OVERWRITE_ORACLE_RESULTS:
        write_csv = False
    else:
        print(f'OVERWRITE_ORACLE_RESULTS=True; overwriting {summary_csv}')
if write_csv:
    with summary_csv.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(summary_rows)
    print('Saved:', summary_csv)
else:
    print('Kept existing:', summary_csv)

summary_payload = {
    'instance_id': INSTANCE_ID,
    'harness_version': HARNESS_VERSION,
    'condition': CONDITION,
    'base_commit': ORACLE_BASE_COMMIT,
    'rows': summary_rows,
}
if write_json_once(summary_json, summary_payload):
    print('Saved:', summary_json)
else:
    print('Kept existing:', summary_json)


## Interpretation boundary

The files written above are prototype CPU-oracle outputs for pipeline validation. A resolved value is formal only for this prototype run and is not a publication-quality SWE-bench score until the same patches are rerun with the official SWE-bench evaluation harness and Docker environment.